In [151]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

# directory that contains agent_tools.py (must run before importing agent_tools)
_pkg = Path.cwd()
# print(_pkg)
if (_pkg / "agent_tools.py").is_file():
    pkg = _pkg
else:
    pkg = _pkg.parent  # typical: cwd is .../notebook, module is one level up

sys.path.insert(0, str(pkg.resolve()))

# Drop stale cache so edits to agent_tools.py are picked up without restarting the kernel
sys.modules.pop("agent_tools", None)
import agent_tools as at
from agent_tools import AgentMemory


In [152]:
df = pd.read_parquet("..\data\heloc_dataset_v1.parquet")

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\arifn\AppData\Local\Temp\ipykernel_17440\4115570229.py:1: SyntaxWarning: invalid escape sequence '\d'
  df = pd.read_parquet("..\data\heloc_dataset_v1.parquet")


In [153]:
df.head()

,RiskPerformance,ExternalRiskEstimate,MSinceOldestTradeOpen,MSinceMostRecentTradeOpen,AverageMInFile,NumSatisfactoryTrades,NumTrades60Ever2DerogPubRec,NumTrades90Ever2DerogPubRec,PercentTradesNeverDelq,MSinceMostRecentDelq,...,NumInqLast6M,NumInqLast6Mexcl7days,NetFractionRevolvingBurden,NetFractionInstallBurden,NumRevolvingTradesWBalance,NumInstallTradesWBalance,NumBank2NatlTradesWHighUtilization,PercentTradesWBalance,label,date
0,Bad,55,144,4,84,20,3,0,83,2,...,0,0,33,-8,8,1,1,69,0,2015-01-01
1,Bad,61,58,15,41,2,4,4,100,-7,...,0,0,0,-8,0,-8,-8,0,0,2015-01-01
2,Bad,67,66,5,24,9,0,0,100,-7,...,4,4,53,66,4,2,1,86,0,2015-01-01
3,Bad,66,169,1,73,28,1,1,93,76,...,5,4,72,83,6,4,3,91,0,2015-01-01
4,Bad,81,333,27,132,12,0,0,100,-7,...,1,1,51,89,3,1,0,80,0,2015-01-01


# Metadata

In [154]:
df.columns

Index(['RiskPerformance', 'ExternalRiskEstimate', 'MSinceOldestTradeOpen',
       'MSinceMostRecentTradeOpen', 'AverageMInFile', 'NumSatisfactoryTrades',
       'NumTrades60Ever2DerogPubRec', 'NumTrades90Ever2DerogPubRec',
       'PercentTradesNeverDelq', 'MSinceMostRecentDelq',
       'MaxDelq2PublicRecLast12M', 'MaxDelqEver', 'NumTotalTrades',
       'NumTradesOpeninLast12M', 'PercentInstallTrades',
       'MSinceMostRecentInqexcl7days', 'NumInqLast6M', 'NumInqLast6Mexcl7days',
       'NetFractionRevolvingBurden', 'NetFractionInstallBurden',
       'NumRevolvingTradesWBalance', 'NumInstallTradesWBalance',
       'NumBank2NatlTradesWHighUtilization', 'PercentTradesWBalance', 'label',
       'date'],
      dtype='str')

In [155]:
am = AgentMemory()

In [156]:
am.col_time = 'date'
am.col_target = 'label'
am.cols_feat = ['ExternalRiskEstimate', 'MSinceOldestTradeOpen',
       'MSinceMostRecentTradeOpen', 'AverageMInFile', 'NumSatisfactoryTrades',
       'NumTrades60Ever2DerogPubRec', 'NumTrades90Ever2DerogPubRec',
       'PercentTradesNeverDelq', 'MSinceMostRecentDelq',
       'MaxDelq2PublicRecLast12M', 'MaxDelqEver', 'NumTotalTrades',
       'NumTradesOpeninLast12M', 'PercentInstallTrades',
       'MSinceMostRecentInqexcl7days', 'NumInqLast6M', 'NumInqLast6Mexcl7days',
       'NetFractionRevolvingBurden', 'NetFractionInstallBurden',
       'NumRevolvingTradesWBalance', 'NumInstallTradesWBalance',
       'NumBank2NatlTradesWHighUtilization', 'PercentTradesWBalance']
am.col_day = 'day'
am.col_month = 'month'
am.col_type = 'type'
am.col_score = 'score'

In [157]:
df[am.col_day] = at.get_day_from_date(df, am.col_time)
df[am.col_month] = at.get_month_from_date(df, am.col_time)
am.cols_feat_num, am.cols_feat_Cat, am.cols_feat_time = at.get_feature_dtype(df, am.cols_feat)


# EDA

In [158]:
am.df_nan_rate = at.get_nan_rate(df, am.cols_feat)
display(am.df_nan_rate)
am.df_nan_rate_timely = at.get_nan_rate_timely(df, am.cols_feat, am.col_month)
display(am.df_nan_rate_timely)

,feature_name,missing_rate
0,ExternalRiskEstimate,0.0
1,MSinceOldestTradeOpen,0.0
2,MSinceMostRecentTradeOpen,0.0
3,AverageMInFile,0.0
4,NumSatisfactoryTrades,0.0
5,NumTrades60Ever2DerogPubRec,0.0
6,NumTrades90Ever2DerogPubRec,0.0
7,PercentTradesNeverDelq,0.0
8,MSinceMostRecentDelq,0.0
9,MaxDelq2PublicRecLast12M,0.0


missing_rate                              \
time_period                              201501 201502 201503 201504 201505   
feature_name                                                                  
AverageMInFile                              0.0    0.0    0.0    0.0    0.0   
ExternalRiskEstimate                        0.0    0.0    0.0    0.0    0.0   
MSinceMostRecentDelq                        0.0    0.0    0.0    0.0    0.0   
MSinceMostRecentInqexcl7days                0.0    0.0    0.0    0.0    0.0   
MSinceMostRecentTradeOpen                   0.0    0.0    0.0    0.0    0.0   
MSinceOldestTradeOpen                       0.0    0.0    0.0    0.0    0.0   
MaxDelq2PublicRecLast12M                    0.0    0.0    0.0    0.0    0.0   
MaxDelqEver                                 0.0    0.0    0.0    0.0    0.0   
NetFractionInstallBurden                    0.0    0.0    0.0    0.0    0.0   
NetFractionRevolvingBurden                  0.0    0.0    0.0    0.0    0.0   
NumBank2NatlTradesWHighUtilization          0.0    0.0    0.0    0.0    0.0   
NumInqLast6M                                0.0    0.0    0.0    0.0    0.0   
NumInqLast6Mexcl7days                       0.0    0.0    0.0    0.0    0.0   
NumInstallTradesWBalance                    0.0    0.0    0.0    0.0    0.0   
NumRevolvingTradesWBalance                  0.0    0.0    0.0    0.0    0.0   
NumSatisfactoryTrades                       0.0    0.0    0.0    0.0    0.0   
NumTotalTrades                              0.0    0.0    0.0    0.0    0.0   
NumTrades60Ever2DerogPubRec                 0.0    0.0    0.0    0.0    0.0   
NumTrades90Ever2DerogPubRec                 0.0    0.0    0.0    0.0    0.0   
NumTradesOpeninLast12M                      0.0    0.0    0.0    0.0    0.0   
PercentInstallTrades                        0.0    0.0    0.0    0.0    0.0   
PercentTradesNeverDelq                      0.0    0.0    0.0    0.0    0.0   
PercentTradesWBalance                       0.0    0.0    0.0    0.0    0.0   

                                                                              \
time_period                        201506 201507 201508 201509 201510 201511   
feature_name                                                                   
AverageMInFile                        0.0    0.0    0.0    0.0    0.0    0.0   
ExternalRiskEstimate                  0.0    0.0    0.0    0.0    0.0    0.0   
MSinceMostRecentDelq                  0.0    0.0    0.0    0.0    0.0    0.0   
MSinceMostRecentInqexcl7days          0.0    0.0    0.0    0.0    0.0    0.0   
MSinceMostRecentTradeOpen             0.0    0.0    0.0    0.0    0.0    0.0   
MSinceOldestTradeOpen                 0.0    0.0    0.0    0.0    0.0    0.0   
MaxDelq2PublicRecLast12M              0.0    0.0    0.0    0.0    0.0    0.0   
MaxDelqEver                           0.0    0.0    0.0    0.0    0.0    0.0   
NetFractionInstallBurden              0.0    0.0    0.0    0.0    0.0    0.0   
NetFractionRevolvingBurden            0.0    0.0    0.0    0.0    0.0    0.0   
NumBank2NatlTradesWHighUtilization    0.0    0.0    0.0    0.0    0.0    0.0   
NumInqLast6M                          0.0    0.0    0.0    0.0    0.0    0.0   
NumInqLast6Mexcl7days                 0.0    0.0    0.0    0.0    0.0    0.0   
NumInstallTradesWBalance              0.0    0.0    0.0    0.0    0.0    0.0   
NumRevolvingTradesWBalance            0.0    0.0    0.0    0.0    0.0    0.0   
NumSatisfactoryTrades                 0.0    0.0    0.0    0.0    0.0    0.0   
NumTotalTrades                        0.0    0.0    0.0    0.0    0.0    0.0   
NumTrades60Ever2DerogPubRec           0.0    0.0    0.0    0.0    0.0    0.0   
NumTrades90Ever2DerogPubRec           0.0    0.0    0.0    0.0    0.0    0.0   
NumTradesOpeninLast12M                0.0    0.0    0.0    0.0    0.0    0.0   
PercentInstallTrades                  0.0    0.0    0.0    0.0    0.0    0.0   
PercentTradesNeverDelq                0.0    0.0    0.0    0

In [159]:
am.df_target_rate_timely = at.get_timely_binary_target_rate(df, am.col_month, am.col_target)

In [160]:
am.df_target_rate_timely

,mean,count
month,,
201501,0.480534,899
201502,0.396552,812
201503,0.353726,899
201504,0.480460,870
201505,0.583982,899
201506,0.489655,870
201507,0.474972,899
201508,0.527374,895
201509,0.386905,840


# Split Data

In [161]:
am.oot_th = 201510
am.hoot_th = 201503

In [162]:
# Use the same integer scale for `col_period` and for oot_th / hoot_th (here: `month`).
df[am.col_type] = at.split_data(df, am.col_target, am.col_month, am.oot_th, am.hoot_th)

In [163]:
df[am.col_type].value_counts()

type
train    2636
hoot     2610
oot      2576
test     1319
valid    1318
Name: count, dtype: int64

In [164]:
am.df_target_rate_per_sample = at.get_target_rate_sample(df, am.col_type, am.col_target)
display(am.df_target_rate_per_sample)


,count,mean
type,,
hoot,2610,0.410728
oot,2576,0.518245
test,1319,0.492039
train,2636,0.491654
valid,1318,0.491654


# Feature transformation

In [165]:
am.dict_bin, am.bp, am.df_binning_stats, am.binning_feature_issues = at.get_optimal_bin(
    df.loc[df[am.col_type] == "train"], am.cols_feat + ['X'], am.col_target, cols_feat_cat=am.cols_feat_cat
)

In [166]:
am.dict_bin

{'ExternalRiskEstimate': [64.5, 68.5, 70.5, 74.5, 80.5],
 'MSinceOldestTradeOpen': [31.5, 107.5, 134.5, 212.5, 273.5],
 'MSinceMostRecentTradeOpen': [2.5, 3.5, 7.5, 9.5, 16.5],
 'AverageMInFile': [52.5, 65.5, 80.5, 97.5, 137.5],
 'NumSatisfactoryTrades': [5.5, 9.5, 11.5, 19.5, 39.5],
 'NumTrades60Ever2DerogPubRec': [0.5, 1.5, 2.5],
 'NumTrades90Ever2DerogPubRec': [0.5, 1.5],
 'PercentTradesNeverDelq': [63.5, 75.5, 83.5, 90.5, 95.5],
 'MSinceMostRecentDelq': [-3.5, 2.5, 15.5, 22.5, 46.5],
 'MaxDelq2PublicRecLast12M': [1.5, 5.5, 6.5],
 'MaxDelqEver': [3.5, 5.5, 6.5],
 'NumTotalTrades': [8.5, 11.5, 21.5, 24.5, 45.5],
 'NumTradesOpeninLast12M': [0.5, 1.5, 2.5, 3.5, 5.5],
 'PercentInstallTrades': [24.5, 35.5, 42.5, 46.5, 65.5],
 'MSinceMostRecentInqexcl7days': [-7.5, 0.5, 1.5, 7.5, 12.5],
 'NumInqLast6M': [0.5, 1.5, 2.5, 4.5],
 'NumInqLast6Mexcl7days': [0.5, 1.5, 2.5, 4.5],
 'NetFractionRevolvingBurden': [14.5, 26.5, 39.5, 65.5, 75.5],
 'NetFractionInstallBurden': [7.5, 33.5, 61.5, 72.5, 93

In [167]:
display(am.df_binning_stats.sort_values(by='gini_power', ascending=False))

,name,dtype,status,selected,n_bins,iv,js,gini,quality_score,gini_power
11,NumTotalTrades,numerical,OPTIMAL,True,6,0.024596,0.003058,0.079225,0.002017,0.920775
2,MSinceMostRecentTradeOpen,numerical,OPTIMAL,True,6,0.035198,0.004382,0.100193,0.018778,0.899807
20,NumInstallTradesWBalance,numerical,OPTIMAL,True,6,0.042386,0.005271,0.108321,0.00329,0.891679
12,NumTradesOpeninLast12M,numerical,OPTIMAL,True,6,0.048713,0.006021,0.111746,0.010252,0.888254
4,NumSatisfactoryTrades,numerical,OPTIMAL,True,6,0.074214,0.009178,0.136911,0.064841,0.863089
6,NumTrades90Ever2DerogPubRec,numerical,OPTIMAL,True,3,0.126368,0.01552,0.143345,0.148225,0.856655
18,NetFractionInstallBurden,numerical,OPTIMAL,True,6,0.070341,0.008731,0.14392,0.031213,0.85608
19,NumRevolvingTradesWBalance,numerical,OPTIMAL,True,6,0.086865,0.010744,0.158586,0.160678,0.841414
16,NumInqLast6Mexcl7days,numerical,OPTIMAL,True,5,0.113001,0.013863,0.168369,0.106171,0.831631
15,NumInqLast6M,numerical,OPTIMAL,True,5,0.129148,0.015784,0.180347,0.059235,0.819653


In [168]:
X_woe, am.cols_feat_woe, am.features_issue_woe = at.get_woe_from_bp(df, am.cols_feat, am.bp)

In [169]:
X_woe

,ExternalRiskEstimate_woe,MSinceOldestTradeOpen_woe,MSinceMostRecentTradeOpen_woe,AverageMInFile_woe,NumSatisfactoryTrades_woe,NumTrades60Ever2DerogPubRec_woe,NumTrades90Ever2DerogPubRec_woe,PercentTradesNeverDelq_woe,MSinceMostRecentDelq_woe,MaxDelq2PublicRecLast12M_woe,...,PercentInstallTrades_woe,MSinceMostRecentInqexcl7days_woe,NumInqLast6M_woe,NumInqLast6Mexcl7days_woe,NetFractionRevolvingBurden_woe,NetFractionInstallBurden_woe,NumRevolvingTradesWBalance_woe,NumInstallTradesWBalance_woe,NumBank2NatlTradesWHighUtilization_woe,PercentTradesWBalance_woe
0,1.212731,0.026363,0.068578,-0.308196,-0.176618,0.982534,-0.179943,0.980232,1.145268,0.959119,...,0.077839,0.431781,-0.336157,-0.293976,0.204033,-0.212588,0.638483,-0.083471,0.254295,0.133993
1,1.212731,1.022666,-0.132759,0.793657,0.675509,0.982534,0.818584,-0.459261,-0.478819,0.283709,...,0.903738,0.431781,-0.336157,-0.293976,-0.723471,-0.212588,-0.017700,-0.415595,-0.423027,-0.834669
2,0.701260,1.022666,0.068578,0.793657,0.432487,-0.266698,-0.179943,-0.459261,-0.478819,-0.494237,...,0.077839,0.431781,0.425731,0.477439,0.429050,0.193523,-0.147205,-0.043969,0.254295,0.641411
3,0.701260,0.026363,0.107014,-0.051872,-0.176618,0.515046,0.634442,0.202690,-0.371713,0.024784,...,0.586148,0.431781,1.011981,0.477439,1.235124,0.226612,0.173665,0.324450,0.760856,1.109830
4,-1.352153,-0.669376,-0.357437,-0.639523,-0.012768,-0.266698,-0.179943,-0.459261,-0.478819,-0.494237,...,-0.136823,0.431781,0.038934,0.011598,0.429050,0.226612,-0.147205,-0.083471,-0.423027,0.361089
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10454,0.019411,0.526229,0.068578,0.462762,-0.176618,-0.266698,-0.179943,0.202690,-0.371713,0.024784,...,-0.341259,-0.456448,-0.336157,-0.293976,-0.362160,-0.212588,0.173665,-0.043969,-0.423027,1.109830
10455,0.701260,0.026363,-0.357437,-0.051872,0.272780,-0.266698,-0.179943,0.202690,0.089215,0.024784,...,0.047152,-0.044198,0.038934,0.011598,1.495328,-0.048350,-0.390585,-0.043969,0.254295,0.361089
10456,0.019411,0.526229,0.068578,0.462762,-0.012768,0.515046,0.634442,-0.459261,-0.478819,0.024784,...,-0.136823,-0.456448,0.425731,0.477439,-0.723471,-0.212588,0.173665,-0.415595,-0.423027,-0.347168
10457,0.019411,-0.217391,-0.132759,-0.639523,-0.365827,0.802181,0.818584,-0.459261,0.089215,0.024784,...,-0.341259,-0.456448,-0.336157,-0.293976,-0.362160,-0.212588,-0.147205,-0.083471,-0.423027,-0.834669


In [170]:
am.df_binning_tables, am.features_issue_binning_tables = at.get_binning_tables_from_bp(am.bp, am.cols_feat + ['x'])

In [171]:
am.df_binning_tables

,Feature Name,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,ExternalRiskEstimate,"(-inf, 64.50)",676,0.256449,525,151,0.223373,1.212731,0.333839,0.039347
1,ExternalRiskEstimate,"[64.50, 68.50)",364,0.138088,246,118,0.324176,0.70126,0.064889,0.007949
2,ExternalRiskEstimate,"[68.50, 70.50)",170,0.064492,102,68,0.400000,0.372078,0.008800,0.001094
3,ExternalRiskEstimate,"[70.50, 74.50)",341,0.129363,175,166,0.486804,0.019411,0.000049,0.000006
4,ExternalRiskEstimate,"[74.50, 80.50)",450,0.170713,158,292,0.648889,-0.647546,0.069545,0.008544
...,...,...,...,...,...,...,...,...,...,...
190,PercentTradesWBalance,"[84.50, 89.50)",163,0.061836,108,55,0.337423,0.641411,0.024475,0.003008
191,PercentTradesWBalance,"[89.50, inf)",393,0.149090,298,95,0.241730,1.10983,0.165460,0.019682
192,PercentTradesWBalance,Special,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000
193,PercentTradesWBalance,Missing,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000


In [172]:
df[am.cols_feat_woe] = am.bp.transform(df[am.cols_feat], metric="woe")

In [173]:
df[am.cols_feat_woe]

,ExternalRiskEstimate_woe,MSinceOldestTradeOpen_woe,MSinceMostRecentTradeOpen_woe,AverageMInFile_woe,NumSatisfactoryTrades_woe,NumTrades60Ever2DerogPubRec_woe,NumTrades90Ever2DerogPubRec_woe,PercentTradesNeverDelq_woe,MSinceMostRecentDelq_woe,MaxDelq2PublicRecLast12M_woe,...,PercentInstallTrades_woe,MSinceMostRecentInqexcl7days_woe,NumInqLast6M_woe,NumInqLast6Mexcl7days_woe,NetFractionRevolvingBurden_woe,NetFractionInstallBurden_woe,NumRevolvingTradesWBalance_woe,NumInstallTradesWBalance_woe,NumBank2NatlTradesWHighUtilization_woe,PercentTradesWBalance_woe
0,1.212731,0.026363,0.068578,-0.308196,-0.176618,0.982534,-0.179943,0.980232,1.145268,0.959119,...,0.077839,0.431781,-0.336157,-0.293976,0.204033,-0.212588,0.638483,-0.083471,0.254295,0.133993
1,1.212731,1.022666,-0.132759,0.793657,0.675509,0.982534,0.818584,-0.459261,-0.478819,0.283709,...,0.903738,0.431781,-0.336157,-0.293976,-0.723471,-0.212588,-0.017700,-0.415595,-0.423027,-0.834669
2,0.701260,1.022666,0.068578,0.793657,0.432487,-0.266698,-0.179943,-0.459261,-0.478819,-0.494237,...,0.077839,0.431781,0.425731,0.477439,0.429050,0.193523,-0.147205,-0.043969,0.254295,0.641411
3,0.701260,0.026363,0.107014,-0.051872,-0.176618,0.515046,0.634442,0.202690,-0.371713,0.024784,...,0.586148,0.431781,1.011981,0.477439,1.235124,0.226612,0.173665,0.324450,0.760856,1.109830
4,-1.352153,-0.669376,-0.357437,-0.639523,-0.012768,-0.266698,-0.179943,-0.459261,-0.478819,-0.494237,...,-0.136823,0.431781,0.038934,0.011598,0.429050,0.226612,-0.147205,-0.083471,-0.423027,0.361089
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10454,0.019411,0.526229,0.068578,0.462762,-0.176618,-0.266698,-0.179943,0.202690,-0.371713,0.024784,...,-0.341259,-0.456448,-0.336157,-0.293976,-0.362160,-0.212588,0.173665,-0.043969,-0.423027,1.109830
10455,0.701260,0.026363,-0.357437,-0.051872,0.272780,-0.266698,-0.179943,0.202690,0.089215,0.024784,...,0.047152,-0.044198,0.038934,0.011598,1.495328,-0.048350,-0.390585,-0.043969,0.254295,0.361089
10456,0.019411,0.526229,0.068578,0.462762,-0.012768,0.515046,0.634442,-0.459261,-0.478819,0.024784,...,-0.136823,-0.456448,0.425731,0.477439,-0.723471,-0.212588,0.173665,-0.415595,-0.423027,-0.347168
10457,0.019411,-0.217391,-0.132759,-0.639523,-0.365827,0.802181,0.818584,-0.459261,0.089215,0.024784,...,-0.341259,-0.456448,-0.336157,-0.293976,-0.362160,-0.212588,-0.147205,-0.083471,-0.423027,-0.834669


# Feature Selection

In [174]:
selected_feats, feats_stats = at.select_features_auc_max_corr(df.loc[df[am.col_type]=='train'], am.cols_feat_woe, am.col_target)

In [175]:
feats_stats

,feature_name,max_correlation,auc_roc,standardized_auc_roc
0,ExternalRiskEstimate_woe,0.493185,0.252408,0.747592
1,PercentTradesWBalance_woe,0.409334,0.334096,0.665904
2,AverageMInFile_woe,0.335007,0.351952,0.648048
3,MSinceMostRecentInqexcl7days_woe,0.349456,0.358304,0.641696
4,NumBank2NatlTradesWHighUtilization_woe,0.493185,0.376375,0.623625
5,NumTrades60Ever2DerogPubRec_woe,0.446790,0.402545,0.597455
6,PercentInstallTrades_woe,0.409334,0.403380,0.596620
7,NumInqLast6M_woe,0.349456,0.409826,0.590174
8,NumRevolvingTradesWBalance_woe,0.469219,0.420707,0.579293
9,NetFractionInstallBurden_woe,0.375085,0.428040,0.571960


In [176]:
selected_feats, feats_stats = at.select_features_iv_max_corr(df.loc[df[am.col_type]=='train'], am.cols_feat_woe, am.col_target)

In [177]:
feats_stats

,feature_name,max_correlation,iv,auc_roc,standardized_auc_roc
0,ExternalRiskEstimate_woe,0.493185,0.809552,0.252408,0.747592
1,PercentTradesWBalance_woe,0.409334,0.342842,0.334096,0.665904
2,MSinceMostRecentInqexcl7days_woe,0.349456,0.302198,0.358304,0.641696
3,AverageMInFile_woe,0.335007,0.286387,0.351952,0.648048
4,NumBank2NatlTradesWHighUtilization_woe,0.493185,0.150515,0.376375,0.623625
5,PercentInstallTrades_woe,0.409334,0.120870,0.403380,0.596620
6,NumTrades60Ever2DerogPubRec_woe,0.446790,0.101090,0.402545,0.597455
7,NumInqLast6M_woe,0.349456,0.093200,0.409826,0.590174
8,NumRevolvingTradesWBalance_woe,0.469219,0.077814,0.420707,0.579293
9,NumSatisfactoryTrades_woe,0.399495,0.069875,0.431545,0.568455
